# Qwen3.5-0.8B 日本語強化学習 (QLoRA / Unsloth)

本ノートブックは **Qwen3.5-0.8B (FP16)** を **llm-jp/oasst1-21k-ja** から抽出した
高品質な日本語指示データ 8,000 件を用いて **QLoRA** 微調整し、
日本語性能を底上げしつつ **破滅的忘却 (catastrophic forgetting)** を最小限に抑えることを目的とします。

## 設計の要点 (忘却抑制)

| 項目 | 設定 | 忘却抑制への寄与 |
|------|------|------------------|
| 手法 | QLoRA (4bit) | 元の重みを凍結し、低ランク差分のみ学習するため原本知識が保持される |
| ターゲット層 | `q/k/v/o/gate/up/down_proj` (全 attn + 全 MLP) | 広く適応させつつ、ランク 16 で表現力を制限 |
| LoRA Rank / Alpha | 16 / 32 (`alpha = 2r`) | ランクを控えめに保ち、過学習を防止 |
| LR | `2e-4` (cosine + warmup) | LoRA 向きの中程度の学習率。フル FT に比べ大幅に低い |
| Epochs | 3 | 過学習・忘却を防ぐため短めに設定 |
| データ | oasst1-21k-ja から高品質 8,000 件 | 多様性と品質を両立させ、特定パターンへの偏りを回避 |
| Optimizer | `paged_adamw_8bit` | VRAM 節約しつつ Adam 系の安定した収束 |
| Gradient Checkpointing | 有効 (Unsloth) | 長文脈 (2048) 学習を Colab T4 でも実行可能 |

## Qwen3.5 特有の注意点

- Qwen3.5-0.8B は **Hybrid 構造** (Gated DeltaNet + Gated Attention + Sparse MoE ではなく dense) の
  マルチモーダルモデルとして公開されていますが、Unsloth 公式サポート済みです。
- 公式推奨は **bf16 LoRA** (VRAM 3GB) ですが、0.8B は dense なので QLoRA (4bit) も動作します。
- テキストのみ学習するため `FastVisionModel.get_peft_model` で `finetune_vision_layers=False` を指定します。
- **Reasoning 保持**: Qwen3.5 は思考プロセス (`<think>...</think>`) を持つため、
  推論能力を保ちたい場合はデータに直接回答と推論スタイルを混ぜる (推論75%以上) ことを推奨します。
- 詳細: https://unsloth.ai/docs/models/qwen3.5/fine-tune


## 1. 環境セットアップ

Unsloth 公式ドキュメントに沿ったインストール手順です (Colab 向け)。
GPU ランタイム (T4 以上推奨) を選択してから実行してください。

In [ ]:
# GPU 確認 (T4 16GB 以上を想定)
!nvidia-smi

In [ ]:
# Unsloth 公式インストール (Colab 用)
# 参考: https://unsloth.ai/docs/models/qwen3.5/fine-tune
import os
if "COLAB_GPU" in os.environ:
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
    !pip install -q xformers==0.0.23.post1 triton==2.2.0
    print("Unsloth インストール完了 (Colab)")
else:
    print("Colab 環境ではありません。ローカルの場合は公式ドキュメントの手順で Unsloth をインストールしてください:")
    print("  https://unsloth.ai/docs/basics/install-to-own-environment")

In [ ]:
# インポート & コンフィグ
import os
import random
import torch
import numpy as np
from datasets import load_dataset, Dataset
from unsloth import FastVisionModel   # Qwen3.5 はマルチモーダル扱いのため FastVisionModel を使用
from trl import SFTTrainer, SFTConfig

# 再現性
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ===== ハイパーパラメータ (仕様書準拠) =====
MODEL_NAME      = "Qwen/Qwen3.5-0.8B"      # 実在確認済 (https://huggingface.co/Qwen/Qwen3.5-0.8B)
MAX_SEQ_LEN     = 2048
LORA_R          = 16
LORA_ALPHA      = 32     # 仕様書準拠 (alpha = 2r)
LORA_DROPOUT    = 0
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]
LR              = 2e-4
EPOCHS          = 3
GRAD_ACCUM      = 4
BATCH_SIZE      = 2          # T4 16GB + 4bit なら bs=2 で 2048 は余裕
WARMUP_STEPS    = 50
DATASET_TARGET  = 8000       # oasst1-21k-ja から抽出する件数
OUTPUT_DIR      = "outputs"
ADAPTER_DIR     = "lora_adapter"

print(f"Model: {MODEL_NAME}")
print(f"LoRA r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}")
print(f"LR={LR} epochs={EPOCHS} grad_accum={GRAD_ACCUM} bs={BATCH_SIZE}")

## 2. モデル読み込み (Unsloth + 4bit QLoRA)

Qwen3.5-0.8B はマルチモーダルモデルとして公開されているため、Unsloth の `FastVisionModel` を使用します。
テキストのみ学習するため、後ほど `finetune_vision_layers=False` を指定します。
`load_in_4bit=True` で QLoRA (4bit 量子化) を有効化します。

In [ ]:
# Qwen3.5-0.8B を 4bit 量子化でロード
model, tokenizer = FastVisionModel.from_pretrained(
    model_name              = MODEL_NAME,
    max_seq_length          = MAX_SEQ_LEN,
    dtype                   = torch.float16,   # 仕様: FP16
    load_in_4bit            = True,            # QLoRA
    use_gradient_checkpointing = "unsloth",
)
print(f"\n[Loaded] vocab={tokenizer.vocab_size}, hidden={model.config.hidden_size}")

In [ ]:
# チャットテンプレートが無ければ Qwen 標準 ChatML を設定
if tokenizer.chat_template is None:
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "chatml",   # Qwen 系標準
        mapping = {"role": "role", "content": "content",
                   "user": "user", "assistant": "assistant"},
    )
    print("chat_template を ChatML として設定しました")
else:
    print("既存 chat_template を使用します")

## 3. LoRA 適用 (ターゲット層: attn 全部 + MLP 全部 / vision は OFF)

仕様書の通り `q/k/v/o/gate/up/down` の 7 モジュールすべてを LoRA ターゲットにします。
`finetune_vision_layers=False` で視覚エンコーダを完全に凍結し、言語層のみ学習します。
これにより VRAM 使用量を抑えつつ、原本の視覚能力を保持します。

In [ ]:
# Qwen3.5 は vision モデル扱いなので FastVisionModel.get_peft_model を使用
# finetune_vision_layers=False でテキストのみ学習
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # ★ テキストのみ学習 (vision は凍結)
    finetune_language_layers   = True,   # 言語層は学習
    finetune_attention_modules = True,   # attn (q/k/v/o) を学習
    finetune_mlp_modules       = True,   # MLP (gate/up/down) を学習
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
    target_modules             = TARGET_MODULES,   # 明示指定 (仕様書準拠)
)

# 学習可能パラメータ数を表示
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. データセット準備 (llm-jp/oasst1-21k-ja → 高品質 8,000 件)

### 実スキーマ (確認済)

`llm-jp/oasst1-21k-ja` は以下の ShareGPT 形式です:

```json
{
  "conversations": [
    {"from": "human", "value": "こんにちは！"},
    {"from": "gpt",   "value": "こんにちは！ご質問や..."},
    ...
  ]
}
```

- 列は `conversations` のみ (rank / message_id / parent_id は存在しない)
- 行数は 21,200 件
- 各行は 1 対話 (turn 数は 2〜6 程度)

### 抽出方針

1. **完全性**: human → gpt → human → gpt のようにペアが取れる会話のみ
2. **長さ**: 各ターンの応答が 50〜4000 文字 (極端値除外)
3. **日本語率**: ひらがな/カタカナ/漢字の割合が 30% 以上 (機械翻訳ノイズ除外)
4. **多様性**: シャッフル後 8,000 件をサンプリング

> 破滅的忘却の観点から **多様性の確保** が極めて重要です。
> 同一トピックへの偏りを防ぐためシャッフルでランダムサンプリングします。

In [ ]:
# データセット読み込み
raw = load_dataset("llm-jp/oasst1-21k-ja", split="train")
print(f"Total rows: {len(raw)}")
print("Columns:", raw.column_names)
print("\n--- Sample (1行目) ---")
import json as _json
print(_json.dumps(raw[0], ensure_ascii=False, indent=2)[:800])

In [ ]:
# 日本語率の簡易チェック
import re
_JP_RE = re.compile(r"[ぁ-んァ-ヶー一-龥]")

def jp_ratio(text: str) -> float:
    if not text:
        return 0.0
    jp_chars = len(_JP_RE.findall(text))
    return jp_chars / max(len(text), 1)

# 品質フィルタ
def is_high_quality(ex) -> bool:
    convs = ex.get("conversations") or []
    if len(convs) < 2:
        return False
    # human と gpt が交互に登場することを確認
    for i, c in enumerate(convs):
        expected = "human" if i % 2 == 0 else "gpt"
        if c.get("from") != expected:
            return False
    # 最初の human 発話と最初の gpt 応答の長さ・日本語率をチェック
    first_user = convs[0].get("value", "")
    first_asst = convs[1].get("value", "")
    if not (20 <= len(first_user) <= 2000):
        return False
    if not (50 <= len(first_asst) <= 4000):
        return False
    if jp_ratio(first_user) < 0.20:
        return False
    if jp_ratio(first_asst) < 0.30:
        return False
    return True

filtered = raw.filter(is_high_quality)
print(f"After quality filter: {len(filtered)}")

In [ ]:
# シャッフル & 8,000 件サンプリング
filtered = filtered.shuffle(seed=SEED)
N = min(DATASET_TARGET, len(filtered))
sampled = filtered.select(range(N))
print(f"Final sampled: {len(sampled)}")
print("\n--- Sample ---")
print(_json.dumps(sampled[0], ensure_ascii=False, indent=2)[:600])

In [ ]:
# ChatML 形式でトークン列を構築
# oasst1 の conversations は human/gpt 形式なので user/assistant にマッピング
ROLE_MAP = {"human": "user", "gpt": "assistant"}

def to_chat_text(ex) -> str:
    msgs = [{"role": ROLE_MAP.get(c["from"], c["from"]), "content": c["value"]}
            for c in ex["conversations"]]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

texts = [to_chat_text(ex) for ex in sampled]
print("=== Sample formatted text ===")
print(texts[0][:800])

print("\n=== Token length stats (first 500) ===")
lens = [len(tokenizer.encode(t, add_special_tokens=False)) for t in texts[:500]]
print(f"min={min(lens)} p50={int(np.percentile(lens,50))} p95={int(np.percentile(lens,95))} max={max(lens)}")

In [ ]:
# Dataset オブジェクト化
train_ds = Dataset.from_dict({"text": texts})
print(train_ds)

## 5. 学習

`paged_adamw_8bit` オプティマイザで VRAM を節約しつつ、
cosine スケジュール + warmup で学習を安定化させます。
`save_strategy="epoch"` で各 epoch のアダプタを保存します。

> **注意**: Qwen3.5 は `<think>...</think>` 推論ブロックを持つモデルです。
> oasst1 には思考プロセスが含まれないため、学習後に推論能力が変化する可能性があります。
> 推論能力を保持したい場合は、推論スタイルのデータ (例: Open-R1 等) を 25% 以上混ぜることを推奨します。

In [ ]:
# 学習モードへ切替 (Vision モデル用)
FastVisionModel.for_training(model)

# SFTConfig (仕様書準拠)
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps                = WARMUP_STEPS,
    num_train_epochs            = EPOCHS,
    learning_rate               = LR,
    fp16                        = True,
    logging_steps               = 10,
    optim                       = "paged_adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = SEED,
    save_strategy               = "epoch",
    save_total_limit            = EPOCHS,
    report_to                   = "none",
    max_grad_norm               = 1.0,
    max_seq_length              = MAX_SEQ_LEN,
    dataset_text_field          = "text",
    packing                     = False,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    args          = training_args,
)

trainer_stats = trainer.train()
print(f"\nTotal steps: {trainer_stats.global_step}")
print(f"Train loss (final): {trainer_stats.training_loss:.4f}")

In [ ]:
# 学習曲線を簡易表示
import pandas as pd
log_df = pd.DataFrame(trainer.state.log_history)
log_df = log_df[log_df["loss"].notna()]
print(log_df.tail(10))

## 6. 保存

LoRA アダプタのみ保存し、マージ済みモデルは別途保存します。
Colab の Google Drive に退避する場合は、右側フォルダアイコンからマウントしてください。

In [ ]:
# LoRA アダプタのみ (軽量)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter to {ADAPTER_DIR}/")

In [ ]:
# (任意) 16bit でマージして保存 (推論用・HF Hub アップロード用)
# Colab の無料 T4 でも 0.8B なら 16bit マージは問題なく動作します
MERGED_DIR = "merged_model_fp16"
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Saved merged 16bit model to {MERGED_DIR}/")

## 7. 推論テスト (日本語 + 英語での忘却確認)

破滅的忘却が起きていないか確認するため、
- 日本語の指示追従性 (向上しているべき)
- 英語での基本的な質問応答 (低下していないべき)

の両方をサンプリングして比較します。

In [ ]:
FastVisionModel.for_inference(model)  # 推論モード

def ask(prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return decoded.strip()

# 日本語テスト
jp_prompts = [
    "日本の四季について短く説明してください。",
    "Python のリスト内包表記の利点を教えてください。",
    "健康的な朝食のアイデアを 3 つ提案してください。",
]
print("=== 日本語推論 ===")
for p in jp_prompts:
    print(f"\n[Q] {p}")
    print(f"[A] {ask(p)}")

In [ ]:
# 英語テスト (忘却チェック)
en_prompts = [
    "Explain the difference between TCP and UDP in one paragraph.",
    "What is the capital of France?",
    "Write a haiku about the ocean.",
]
print("=== English inference (forgetting check) ===")
for p in en_prompts:
    print(f"\n[Q] {p}")
    print(f"[A] {ask(p)}")

## 8. (任意) Hugging Face Hub へアップロード

学習したアダプタを公開・共有する場合の例です。
Colab シークレットに `HF_TOKEN` を登録してからコメントアウトを外してください。

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))
#
# HF_REPO = "your-username/qwen3.5-0.8b-ja-qlora-oasst1-8k"
# model.push_to_hub_merged(HF_REPO, tokenizer, save_method="merged_16bit")
# model.push_to_hub(HF_REPO)  # adapter のみ

## 9. 今後の改善方向

1. **評価の定量化和**:
   - `llm-jp-eval` / `jp-eval` で日本語タスクのスコアを比較
   - `lm-evaluation-harness` で MMLU / MMLU-JA を実行
2. **混合データによる忘却抑制の強化**:
   - 英語汎用データ (例: 1k 件の多言語 Wikipedia) を 10% ブレンドすると更に安定
3. **対話継続性能の強化**:
   - oasst1 は単発対話中心なので、マルチターンデータ (`kunishou/databricks-dolly-15k-ja` 等) を追加
4. **DPO / ORPO による好みチューニング**:
   - 本 SFT のあとに DPO を載せると、応答の自然さが向上する場合があります
5. **Reasoning 保持**:
   - Qwen3.5 の `<think>...</think>` 推論能力を保ちたい場合は、
     推論スタイルデータを 25% 以上混ぜる (Unsloth 公式推奨)

詳細は `ROADMAP.md` を参照してください。